# Import Required Libraries
Import the necessary libraries, including h5py and numpy.

In [14]:
# Import Required Libraries
import pandas as pd
import h5py
import numpy as np
import nibabel as nib
import os
from PIL import Image
import numpy as np
# h5py is used for handling HDF5 files
# numpy is used for numerical operations

# Load h5 File
Use h5py to load an h5 file from a specified path.

In [4]:
ls


'=1.26.0,'                     rename.sh*
 bibtex.bib                    requirements_python3.6.txt
 brats_labels.csv              requirements_python3.8.txt
 brats_training.yaml           results/
 build/                        run_freeview.sh*
 data/                         run.ipynb
 data_processing.ipynb         run_pred.sh*
 dist/                         scripts/
 ext/                          setup.py
 extract_first_channel.sh*     SynthSeg/
 freeview.sh*                  SynthSeg.egg-info/
 gen_vs_output.gif             test.ipynb
 ground_truth_vs_predict.gif   training_labels.csv
 LICENSE.txt                   training_log_20250416_235405.txt
 models/                       training_log_20250416_235420.txt
 move_files.sh*                training_log_20250416_235432.txt
 new_class.sh*                 training_sample1.png
 output.log                    training_seg_01.nif
 predict_log.txt               view_brats.sh*
 README.md


In [37]:
image = nib.load("data/Brats/images/volume_1.nii.gz")
image

In [32]:
mask =  nib.load("data/Brats/masks/volume_1_mask.nii.gz").get_fdata()

In [38]:
image.dataobj.dtype

dtype('<f4')

In [33]:
mask.dtype

dtype('float64')

In [3]:
# Load h5 File
file_path = 'data/BraTS2020_training_data/content/data/volume_1_slice_0.h5'  # specify the path to your h5 file

# Open the h5 file in read mode
with h5py.File(file_path, 'r') as h5_file:
    # Explore the structure of the H5 file
    def print_structure(name, obj):
        print(name)
    h5_file.visititems(print_structure)
    
    # Access the 'image' dataset
    if 'image' in h5_file:
        image_data = h5_file['image']
        print("Image shape:", image_data.shape)
    else:
        print("Dataset 'image' not found in the file.")
    
    # Access the 'mask' dataset
    if 'mask' in h5_file:
        mask_data = h5_file['mask']
        print("Mask shape:", mask_data.shape)
    else:
        print("Dataset 'mask' not found in the file.")

image
mask
Image shape: (240, 240, 4)
Mask shape: (240, 240, 3)


In [ ]:

# Directory containing the images
main_dir = 'data/NINS_Dataset/'
image_dir = 'Brain Atrophy'

# Initialize a list to store the images
image_list = []

# Iterate over all files in the directory
for filename in os.listdir(main_dir + image_dir):
    if filename.endswith('.png') or filename.endswith('.jpg'):  # Adjust the file extensions as needed
        image_path = os.path.join(main_dir + image_dir, filename)
        image = Image.open(image_path)
        image = image.resize((240, 240), Image.ANTIALIAS)
        image = image.convert('L')
        image_array = np.array(image)
        
        image_list.append(image_array)

# Stack images along a new dimension
stacked_images = np.stack(image_list, axis=0)

print("Stacked images shape:", stacked_images.shape)
out_path = 'data/NINS/' + image_dir + '.nii'
image_nii = nib.Nifti1Image(stacked_images, np.eye(4))
nib.save(image_nii, out_path)

FileNotFoundError: [Errno 2] No such file or directory: 'data/NINS_Dataset/Brain Atrophy'

In [52]:

# Load the CSV file
csv_path = 'data/BraTS2020_training_data/content/data/meta_data.csv'
df = pd.read_csv(csv_path)

# Group by volume to process each volume separately
grouped = df.groupby('volume')

for volume_id, group in grouped:
    # Initialize lists to store image and mask slices
    image_slices = []
    mask_slices = []

    for _, row in group.iterrows():
        h5_path = "data/BraTS2020_training_data" + row['slice_path']
        
        with h5py.File(h5_path, 'r') as h5_file:
            # Assuming the datasets are named 'image' and 'mask'
            img = h5_file['image'][:]  # shape (240, 240, 4)
            msk = h5_file['mask'][:]   # shape (240, 240, 3)
            # Resize images and masks to (160, 160)
            img_resized = np.stack([np.array(Image.fromarray(img[..., i]).resize((160, 160), Image.BILINEAR)) for i in range(img.shape[-1])], axis=-1)
            msk_resized = np.stack([np.array(Image.fromarray(msk[..., i]).resize((160, 160), Image.NEAREST)) for i in range(msk.shape[-1])], axis=-1)
            
            # Ensure mask is binary (0 or 1)
            msk_resized = np.where(msk_resized > 0.5, 1, 0)
            
            image_slices.append(img_resized)
            mask_slices.append(msk_resized)
        # Stack slices to form 3D volumes
    image_slices = np.array(image_slices)  # shape (num_slices, 160, 160, 4)
    mask_slices = np.array(mask_slices)    # shape (num_slices, 160, 160, 3)
    
    # Transpose to get shape (h, w, num_slices, 4) for images and (h, w, num_slices, 3) for masks
    image_volumes = np.transpose(image_slices, (1, 2, 0, 3))
    mask_volumes = np.transpose(mask_slices, (1, 2, 0, 3))
    image_volumes = image_volumes[..., 0] 
    image_volumes = image_volumes[..., np.newaxis]  # Add a new axis to make it (h, w, num_slices, 1)
    # only one channel
    print(image_volumes.shape, image_volumes.dtype)
    
    # # Save each MRI type and mask type as separate NIfTI images
    # mri_types = ['type1', 'type2', 'type3', 'type4']
    mask_types = ['mask1', 'mask2', 'mask3']
    
    # for i, mri_type in enumerate(mri_types):
    image_nii = nib.Nifti1Image(image_volumes, np.eye(4).astype(np.float32))
    image_nii_path = f'data/Brats/images/volume_{volume_id}.nii.gz'
    nib.save(image_nii, image_nii_path)
    print(f'Saved {image_nii_path}')
    ne_mask = mask_volumes[..., 0] 
    ne_mask = np.where(mask_volumes[...,1] > 0, 2, ne_mask)
    ne_mask = np.where(mask_volumes[...,2] > 0, 3, ne_mask)
    ne_mask = np.where([image_volumes[...,0]> 0], ne_mask + 100, 0)
    ne_mask = ne_mask.squeeze()
    ne_mask = ne_mask.astype(np.int16)
    print(ne_mask.shape, ne_mask.dtype)
    # for j, mask_type in enumerate(mask_types):
    #     new_mask = np.zeros_like(mask_volumes[..., j])
    mask_nii = nib.Nifti1Image(ne_mask, np.eye(4).astype(np.int16))
    mask_nii_path = f'data/Brats/masks/volume_{volume_id}_mask.nii.gz'
    nib.save(mask_nii, mask_nii_path)
    print(f'Saved {mask_nii_path}')


(160, 160, 155, 1) float32
Saved data/Brats/images/volume_1.nii.gz
(160, 160, 155) int16
Saved data/Brats/masks/volume_1_mask.nii.gz
(160, 160, 155, 1) float32
Saved data/Brats/images/volume_2.nii.gz
(160, 160, 155) int16
Saved data/Brats/masks/volume_2_mask.nii.gz
(160, 160, 155, 1) float32
Saved data/Brats/images/volume_3.nii.gz
(160, 160, 155) int16
Saved data/Brats/masks/volume_3_mask.nii.gz
(160, 160, 155, 1) float32
Saved data/Brats/images/volume_4.nii.gz
(160, 160, 155) int16
Saved data/Brats/masks/volume_4_mask.nii.gz
(160, 160, 155, 1) float32
Saved data/Brats/images/volume_5.nii.gz
(160, 160, 155) int16
Saved data/Brats/masks/volume_5_mask.nii.gz
(160, 160, 155, 1) float32
Saved data/Brats/images/volume_6.nii.gz
(160, 160, 155) int16
Saved data/Brats/masks/volume_6_mask.nii.gz
(160, 160, 155, 1) float32
Saved data/Brats/images/volume_7.nii.gz
(160, 160, 155) int16
Saved data/Brats/masks/volume_7_mask.nii.gz
(160, 160, 155, 1) float32
Saved data/Brats/images/volume_8.nii.gz
(1

# Explore h5 File Structure
Explore the structure of the h5 file, including groups and datasets.

In [ ]:
# Explore h5 File Structure

# Open the h5 file in read mode
with h5py.File(file_path, 'r') as h5_file:
    # Function to recursively explore the structure of the h5 file
    def explore_h5_structure(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name}, shape: {obj.shape}, dtype: {obj.dtype}")

    # Visit all items in the file
    h5_file.visititems(explore_h5_structure)

# Extract Data from h5 File
Extract specific datasets from the h5 file and convert them to numpy arrays for further analysis.

In [ ]:
# Extract Data from h5 File

# Open the h5 file in read mode
with h5py.File(file_path, 'r') as h5_file:
    # Extract specific datasets
    dataset_1 = h5_file['dataset_1'][:]  # replace 'dataset_1' with your actual dataset name
    dataset_2 = h5_file['dataset_2'][:]  # replace 'dataset_2' with your actual dataset name

# Convert datasets to numpy arrays
array_1 = np.array(dataset_1)
array_2 = np.array(dataset_2)

# Display the extracted data
print("Dataset 1:", array_1)
print("Dataset 2:", array_2)

In [2]:
import numpy as np

In [ ]:
given_labels = np.load('data/labels_classes_priors/synthseg_segmentation_labels.npy')
label_names = np.load('data/labels_classes_priors/synthseg_segmentation_names_2.0.npy')
denoiser =  np.load('data/labels_classes_priors/synthseg_denoiser_labels_2.0.npy')
topological_classes = np.load('data/labels_classes_priors/synthseg_topological_classes_2.0.npy')
parcellation_labels = np.load('data/labels_classes_priors/synthseg_parcellation_labels.npy')
parcellation_names = np.load('data/labels_classes_priors/synthseg_parcellation_names.npy')
qc_labels = np.load('data/labels_classes_priors/synthseg_qc_labels.npy')
qc_names = np.load('data/labels_classes_priors/synthseg_qc_names.npy')

In [4]:

print( "denoiser", len(denoiser))
print("topological_classes", len(topological_classes))
print("given_labels", len(given_labels))
print("parcellation", len(parcellation_labels))
print("qc_labels", len(qc_labels))

denoiser 55
topological_classes 55
given_labels 55
parcellation 69
qc_labels 54


In [5]:
unique_num = 100
new_label = np.array([101,102,103])
new_names = np.array(['tumor-1', 'tumor-2', 'tumor-3'])
updated_label = np.append(given_labels, new_label)
updated_names = np.append(label_names, new_names)
new_denoiser = np.ones(len(new_label))
updated_denoiser = np.append(denoiser, new_denoiser)
new_topological_classes = np.ones(len(new_label))*unique_num
updated_topological_classes = np.append(topological_classes, new_topological_classes)

In [2]:
import numpy as np

In [40]:
given_labels = np.load('data/labels_classes_priors/synthseg_segmentation_labels_2.0.npy')

In [8]:
given_labels

array([ 0, 14, 15, 16,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  2,  3,  4,  5,  7,  8, 10, 11, 12, 13, 17, 18,  2, 26, 28,  0,
        4,  5, 41, 42, 43, 44, 46, 47, 49, 50, 51, 52, 53, 54, 41, 58, 60,
        0, 43, 44], dtype=int32)

In [14]:
brats_labels = np.load('data/labels_classes_priors/brats_synthseg_segmentation_labels.npy')

In [15]:
len(np.unique(given_labels))

35

In [5]:
import tensorflow as tf
import numpy as np
import h5py

In [6]:
import tensorflow as tf
from tensorflow.keras.models import load_model

In [41]:
model_path = "models/synthseg_2.0.h5"  # Replace with the actual path to your model

In [39]:
brats_labels = np.load('data/labels_classes_priors/synthseg_segmentation_labels.npy')

In [6]:
only_brats_labels = np.array([0,100,101,102,103]).astype(np.int32)
np.save('data/labels_classes_priors/only_brats_labels.npy', only_brats_labels)

In [5]:
brats_labels.dtype

dtype('int32')

In [7]:
[None]*3 + [1]

[None, None, None, 1]

In [41]:
import os
import numpy as np
import tensorflow as tf
from ext.lab2im import layers
from ext.neuron import models as nrn_models

# Define the model path
model_path = "models/synthseg_2.0.h5" # Replace with the actual path

# Define input parameters
input_shape = [None,None,None,1]  # 3D input with 1 channel
labels_segmentation = np.unique(given_labels)  # Example label list
n_levels = 5
nb_conv_per_level = 2
conv_size = 3
unet_feat_count = 24
feat_multiplier = 2
activation = 'elu'
sigma_smoothing = 0
flip_indices = None
gradients = False

# Define the build_model function
def build_model(path_model,
                input_shape,
                labels_segmentation,
                n_levels,
                nb_conv_per_level,
                conv_size,
                unet_feat_count,
                feat_multiplier,
                activation,
                sigma_smoothing,
                flip_indices,
                gradients):
    # assert os.path.isfile(path_model), "The provided model path does not exist."

    # Get the number of labels
    n_labels_seg = len(labels_segmentation)

    # Build the UNet
    net = nrn_models.unet(input_shape=input_shape,
                          nb_labels=n_labels_seg,
                          nb_levels=n_levels,
                          nb_conv_per_level=nb_conv_per_level,
                          conv_size=conv_size,
                          nb_features=unet_feat_count,
                          feat_mult=feat_multiplier,
                          activation=activation,
                          batch_norm=-1)
    if path_model is not None:
        net.load_weights(path_model, by_name=True, skip_mismatch=True)

    # Smooth posteriors if specified
    if sigma_smoothing > 0:
        last_tensor = net.output
        last_tensor = layers.GaussianBlur(sigma=sigma_smoothing)(last_tensor)
        net = tf.keras.models.Model(inputs=net.inputs, outputs=last_tensor)

    return net

# Load the model
model = build_model(model_path,
                    input_shape,
                    labels_segmentation,
                    n_levels,
                    nb_conv_per_level,
                    conv_size,
                    unet_feat_count,
                    feat_multiplier,
                    activation,
                    sigma_smoothing,
                    flip_indices,
                    gradients)

# Print the model summary
model.summary()

Using TensorFlow backend.
2025-05-22 11:31:11.389541: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcuda.so.1
2025-05-22 11:31:11.408940: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-05-22 11:31:11.409194: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1561] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA GeForce RTX 4070 Ti computeCapability: 8.9
coreClock: 2.61GHz coreCount: 60 deviceMemorySize: 11.71GiB deviceMemoryBandwidth: 469.43GiB/s
2025-05-22 11:31:11.409542: W tensorflow/stream_executor/platform/default/dso_loader.cc:55] Could not load dynamic library 'libcudart.so.10.1'; dlerror: libcudart.so.10.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/lib/x86_64-linux-gnu:
2025-05-22 11:31:11.409585: W tensorflow/strea

Model: "unet"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
unet_input (InputLayer)         (None, None, None, N 0                                            
__________________________________________________________________________________________________
unet_conv_downarm_0_0 (Conv3D)  (None, None, None, N 672         unet_input[0][0]                 
__________________________________________________________________________________________________
unet_conv_downarm_0_1 (Conv3D)  (None, None, None, N 15576       unet_conv_downarm_0_0[0][0]      
__________________________________________________________________________________________________
unet_bn_down_0 (BatchNormalizat (None, None, None, N 96          unet_conv_downarm_0_1[0][0]      
_______________________________________________________________________________________________

In [42]:
new_model = build_model(None,
                    [None,None,None,1],
                    np.array([0,100,101,102,103]),
                    n_levels,
                    nb_conv_per_level,
                    conv_size,
                    unet_feat_count,
                    feat_multiplier,
                    activation,
                    sigma_smoothing,
                    flip_indices,
                    gradients)

In [43]:
new_model.summary()

Model: "unet"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
unet_input (InputLayer)         (None, None, None, N 0                                            
__________________________________________________________________________________________________
unet_conv_downarm_0_0 (Conv3D)  (None, None, None, N 672         unet_input[0][0]                 
__________________________________________________________________________________________________
unet_conv_downarm_0_1 (Conv3D)  (None, None, None, N 15576       unet_conv_downarm_0_0[0][0]      
__________________________________________________________________________________________________
unet_bn_down_0 (BatchNormalizat (None, None, None, N 96          unet_conv_downarm_0_1[0][0]      
_______________________________________________________________________________________________

In [44]:
print("Output shape of new_model:", new_model.output_shape)

Output shape of new_model: (None, None, None, None, 5)


In [45]:
new_model.summary()

Model: "unet"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
unet_input (InputLayer)         (None, None, None, N 0                                            
__________________________________________________________________________________________________
unet_conv_downarm_0_0 (Conv3D)  (None, None, None, N 672         unet_input[0][0]                 
__________________________________________________________________________________________________
unet_conv_downarm_0_1 (Conv3D)  (None, None, None, N 15576       unet_conv_downarm_0_0[0][0]      
__________________________________________________________________________________________________
unet_bn_down_0 (BatchNormalizat (None, None, None, N 96          unet_conv_downarm_0_1[0][0]      
_______________________________________________________________________________________________

In [46]:
# Get the second last layer of the new_model
second_last_layer = new_model.layers[-2]

# Print the details of the second last layer
print("Second last layer:", second_last_layer.name)
print("Layer details:", second_last_layer)

Second last layer: unet_likelihood
Layer details: <keras.layers.convolutional.Conv3D object at 0x736a697e7e80>


In [47]:
for layer_new, layer_old in zip(new_model.layers, model.layers):
    if layer_new.name != second_last_layer.name and layer_new.name != "unet_conv_downarm_0_0" :  # Skip the second last layer
        layer_new.set_weights(layer_old.get_weights())
        print(f"Copied weights for layer: {layer_new.name}")
    else:
        print(f"Skipped copying weights for layer: {layer_new.name}")

Copied weights for layer: unet_input
Skipped copying weights for layer: unet_conv_downarm_0_0
Copied weights for layer: unet_conv_downarm_0_1
Copied weights for layer: unet_bn_down_0
Copied weights for layer: unet_maxpool_0
Copied weights for layer: unet_conv_downarm_1_0
Copied weights for layer: unet_conv_downarm_1_1
Copied weights for layer: unet_bn_down_1
Copied weights for layer: unet_maxpool_1
Copied weights for layer: unet_conv_downarm_2_0
Copied weights for layer: unet_conv_downarm_2_1
Copied weights for layer: unet_bn_down_2
Copied weights for layer: unet_maxpool_2
Copied weights for layer: unet_conv_downarm_3_0
Copied weights for layer: unet_conv_downarm_3_1
Copied weights for layer: unet_bn_down_3
Copied weights for layer: unet_maxpool_3
Copied weights for layer: unet_conv_downarm_4_0
Copied weights for layer: unet_conv_downarm_4_1
Copied weights for layer: unet_bn_down_4
Copied weights for layer: unet_up_5
Copied weights for layer: unet_merge_5
Copied weights for layer: unet

In [34]:
# Get the weights and biases of the 'unet_conv_downarm_0_0' layer from the original model
weights_old, biases_old = model.get_layer("unet_conv_downarm_0_0").get_weights()

# Modify the weights to distribute them across 4 channels in the new model
weights_new = np.repeat(weights_old, 4, axis=-2) / 4  # Repeat weights for 4 channels and divide by 4
biases_new = biases_old  # Biases remain the same

# Set the updated weights and biases to the 'unet_conv_downarm_0_0' layer in the new model
new_model.get_layer("unet_conv_downarm_0_0").set_weights([weights_new, biases_new])

print("Updated weights for 'unet_conv_downarm_0_0' layer in the new model.")

Updated weights for 'unet_conv_downarm_0_0' layer in the new model.


In [48]:
new_model.save_weights('models/brats_5ch_3.0.h5')
print("Weights for new_model have been saved to 'new_model_weights.h5'")

Weights for new_model have been saved to 'new_model_weights.h5'


In [9]:
new_model.summary()

Model: "unet"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
unet_input (InputLayer)         (None, None, None, N 0                                            
__________________________________________________________________________________________________
unet_conv_downarm_0_0 (Conv3D)  (None, None, None, N 672         unet_input[0][0]                 
__________________________________________________________________________________________________
unet_conv_downarm_0_1 (Conv3D)  (None, None, None, N 15576       unet_conv_downarm_0_0[0][0]      
__________________________________________________________________________________________________
unet_bn_down_0 (BatchNormalizat (None, None, None, N 96          unet_conv_downarm_0_1[0][0]      
_______________________________________________________________________________________________

In [10]:
model.summary()

Model: "unet"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
unet_input (InputLayer)         (None, None, None, N 0                                            
__________________________________________________________________________________________________
unet_conv_downarm_0_0 (Conv3D)  (None, None, None, N 672         unet_input[0][0]                 
__________________________________________________________________________________________________
unet_conv_downarm_0_1 (Conv3D)  (None, None, None, N 15576       unet_conv_downarm_0_0[0][0]      
__________________________________________________________________________________________________
unet_bn_down_0 (BatchNormalizat (None, None, None, N 96          unet_conv_downarm_0_1[0][0]      
_______________________________________________________________________________________________

In [9]:
import numpy as np

In [10]:
score = np.load("results/brats_seg/dice.npy")

In [11]:
score.shape

(35, 200)

In [12]:
print(score.mean(axis=1))

[7.13820827e-01 7.64021413e-02 5.55576226e-02 1.16897779e-02
 5.42230282e-03 1.74777655e-02 2.63040245e-02 6.50681651e-03
 3.13121726e-03 3.84045184e-03 1.76595675e-03 5.53731779e-03
 5.58356181e-03 2.28919697e-02 9.39561092e-03 7.44951751e-03
 2.90046483e-03 7.89837632e-03 5.66226244e-02 5.22426131e-02
 1.82084241e-02 4.36046409e-03 8.67178044e-03 3.12713989e-02
 1.20212983e-02 3.52999139e-03 5.53123823e-03 3.42477493e-03
 4.59963981e-03 8.74429576e-03 3.05786880e-03 7.61325831e-03
 2.19106045e-04 0.00000000e+00 1.41386033e-04]


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

In [10]:
t_mask = nib.load("data/Brats_resize/masks/volume_006_mask.nii.gz").get_fdata()

In [11]:
np.unique(t_mask)

array([  0., 101., 102., 103.])